# GT Keyword Extraction and Processing Workflow

In [5]:
# Configuration
dataset_name = "indianexpress"
# url_gt = "GT.txt"
# url = "ULR.txt"
dataset_language = "english"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 100

# Stemming and Stopword Libraries (English)
from nltk.stem.snowball import SnowballStemmer
import nltk
from nltk.corpus import stopwords  # Stopword library

# Initialize stemmer and stopwords for English
stemmer = SnowballStemmer(dataset_language)
nltk.download('stopwords')
nltk_english_stopwords = set(stopwords.words(dataset_language))

# Tag rating set
tag_rating_set = {
    "h1": 40.3621,
    "head": 32.4698,
    "title": 32.4698,
    "URL": 30.5916,
    "h2": 22.4907,
    "span": 18.7557,
    "p": 10.2696,
    "div": 7.7747,
}

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/muditha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
import re
def process_keywords(keywords):
    """Apply cleaning, stemming, and stopword filtering to a list of keywords. Returns a processed list of useful words."""
    processed = []
    for word in keywords:
        # Clean word: keep only alphabetic characters
        cleaned_word = re.sub(r'[^a-zA-Z]', '', word)
        if not cleaned_word:
            continue
        # Stem word
        stemmed_word = stemmer.stem(cleaned_word.lower())
        # Filter stopwords
        if stemmed_word not in nltk_english_stopwords:
            processed.append(stemmed_word)
    # Return unique processed keywords (useful words only)
    return processed

In [7]:
from bs4 import BeautifulSoup
def extract_and_process_keywords_from_tag(html_text, tag_name):
    """Extracts keywords from the content inside the specified tag in the given HTML text, processes them using process_keywords, and returns a dictionary with keyword frequencies."""
    soup = BeautifulSoup(html_text, 'html.parser')
    elements = soup.find_all(tag_name)
    raw_keywords = []
    for elem in elements:
        text = elem.get_text(separator=' ')
        raw_keywords.extend(text.split())
    processed = process_keywords(raw_keywords)
    freq = {}
    for kw in processed:
        freq[kw] = freq.get(kw, 0) + 1
    return freq

In [ ]:
# 5. Aggregate and Print URL Ratings Across All Pages (as a pseudo-tag for sorting)
from urllib.parse import urlparse, unquote
from collections import defaultdict
import urllib.request
import matplotlib.pyplot as plt

frequency_weight = 10

# Store metrics for all pages
dataset_precisions = []
dataset_recalls = []
dataset_f1s = []

for i in range(num_pages):
    gt_url = f"{base_url}/{i}/GT.txt"
    web_page_url = f"{base_url}/{i}"
    url_file_url = f"{base_url}/{i}/URL.txt"
    processed_gt_keywords = []
    extracting_keywords = {}
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Copilot/1.0)"}
    gt_req = urllib.request.Request(gt_url, headers=headers)
    web_req = urllib.request.Request(web_page_url, headers=headers)
    url_req = urllib.request.Request(url_file_url, headers=headers)
    try:
        with urllib.request.urlopen(gt_req, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            gt_keywords = gt_text.split()
            processed_gt_keywords = list(set(process_keywords(gt_keywords)))
    except Exception as e:
        continue
    # HTML tag ratings
    try:
        with urllib.request.urlopen(web_req, timeout=5) as web_response:
            html_text = web_response.read().decode("utf-8-sig").strip()

            for tag, rating in tag_rating_set.items():
                result = extract_and_process_keywords_from_tag(html_text, tag)
                for kw, freq in result.items():
                    value = rating + freq * frequency_weight
                    if kw in extracting_keywords:
                        extracting_keywords[kw] += value
                    else:
                        extracting_keywords[kw] = value
    except Exception as e:
        continue
    # URL rating as a pseudo-tag
    try:
        with urllib.request.urlopen(url_req, timeout=5) as url_response:
            real_url = url_response.read().decode("utf-8-sig").strip()
            parsed_url = urlparse(real_url)
            normalized_path = unquote(parsed_url.path.lower())
            url_tokens = re.findall(r"[a-zåäöA-ZÅÄÖ0-9]+", normalized_path)
            processed_url_keywords = process_keywords(url_tokens)
            # Use the same scoring pattern as tags, with 'URL' rating
            url_rating = tag_rating_set.get('URL', 0)
            for kw in processed_url_keywords:
                value = url_rating  # treat each URL keyword as freq=1
                if kw in extracting_keywords:
                    extracting_keywords[kw] += value
                else:
                    extracting_keywords[kw] = value
    except Exception as e:
        continue
    # Calculate Precision, Recall, and F1-score
    top_10 = sorted(extracting_keywords.items(), key=lambda x: x[1], reverse=True)[:10]
    extracted_keywords = set([kw for kw, _ in top_10])
    gt_keywords_set = set(processed_gt_keywords)

    # Calculate metrics
    true_positives = len(extracted_keywords & gt_keywords_set)
    precision = true_positives / len(extracted_keywords) if len(extracted_keywords) > 0 else 0
    recall = true_positives / len(gt_keywords_set) if len(gt_keywords_set) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    dataset_precisions.append(precision)
    dataset_recalls.append(recall)
    dataset_f1s.append(f1_score)

    print(f"Page {i}: Precision={precision:.4f}, Recall={recall:.4f}, F1={f1_score:.4f}")

# Visualize and compare performance
plt.figure(figsize=(12, 6))
plt.plot(dataset_precisions, label='Precision', marker='o')
plt.plot(dataset_recalls, label='Recall', marker='s')
plt.plot(dataset_f1s, label='F1-score', marker='^')
plt.xlabel('Page Index')
plt.ylabel('Score')
plt.title('Keyword Extraction Performance Across Pages')
plt.legend()
plt.grid(True)
plt.show()

# Print average metrics
print(f"\nAverage Precision: {sum(dataset_precisions)/len(dataset_precisions):.4f}")
print(f"Average Recall: {sum(dataset_recalls)/len(dataset_recalls):.4f}")
print(f"Average F1-score: {sum(dataset_f1s)/len(dataset_f1s):.4f}")


Page 0: Precision=0.5000, Recall=1.0000, F1=0.6667
Page 1: Precision=0.3000, Recall=0.6000, F1=0.4000
Page 2: Precision=0.2000, Recall=0.4000, F1=0.2667
Page 3: Precision=0.5000, Recall=1.0000, F1=0.6667
Page 4: Precision=0.8000, Recall=0.7273, F1=0.7619
Page 5: Precision=0.6000, Recall=1.0000, F1=0.7500
Page 6: Precision=0.2000, Recall=0.5000, F1=0.2857
Page 7: Precision=0.4000, Recall=0.4444, F1=0.4211
Page 8: Precision=0.3000, Recall=0.6000, F1=0.4000
Page 9: Precision=0.6000, Recall=0.8571, F1=0.7059
Page 10: Precision=0.2000, Recall=0.2857, F1=0.2353
Page 11: Precision=0.4000, Recall=0.5714, F1=0.4706
Page 12: Precision=0.7000, Recall=0.7778, F1=0.7368
Page 13: Precision=0.4000, Recall=0.6667, F1=0.5000
Page 14: Precision=0.4000, Recall=0.8000, F1=0.5333
Page 15: Precision=0.4000, Recall=0.5714, F1=0.4706
Page 16: Precision=0.4000, Recall=0.8000, F1=0.5333
